In [ ]:
import os, time, random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from torchvision.datasets import GTSRB
from torchvision.models import resnet50, ResNet50_Weights
from sklearn.metrics import confusion_matrix, classification_report

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMG_SIZE = 64
NUM_CLASSES = 43
print('Device:', DEVICE)

In [ ]:
# Download GTSRB dataset
trainval_raw = GTSRB(root='./data', split='train', download=True)
test_raw     = GTSRB(root='./data', split='test',  download=True)

CLASS_NAMES = [
    'Speed limit (20km/h)', 'Speed limit (30km/h)', 'Speed limit (50km/h)',
    'Speed limit (60km/h)', 'Speed limit (70km/h)', 'Speed limit (80km/h)',
    'End of speed limit (80km/h)', 'Speed limit (100km/h)', 'Speed limit (120km/h)',
    'No passing', 'No passing veh > 3.5t', 'Right-of-way at intersection',
    'Priority road', 'Yield', 'Stop', 'No vehicles', 'Veh > 3.5t prohibited',
    'No entry', 'General caution', 'Dangerous curve left', 'Dangerous curve right',
    'Double curve', 'Bumpy road', 'Slippery road', 'Road narrows right', 'Road work',
    'Traffic signals', 'Pedestrians', 'Children crossing', 'Bicycles crossing',
    'Beware ice/snow', 'Wild animals crossing', 'End speed+passing limits',
    'Turn right ahead', 'Turn left ahead', 'Ahead only', 'Go straight or right',
    'Go straight or left', 'Keep right', 'Keep left', 'Roundabout mandatory',
    'End of no passing', 'End no passing veh > 3.5t',
]

# Class distribution
labels_all = [trainval_raw[i][1] for i in range(len(trainval_raw))]
counts = np.bincount(labels_all, minlength=NUM_CLASSES)
class_to_idx = {c: [] for c in range(NUM_CLASSES)}
for idx, lbl in enumerate(labels_all):
    class_to_idx[lbl].append(idx)

print(f'Train+Val: {len(trainval_raw):,}   Test: {len(test_raw):,}')

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(range(NUM_CLASSES), counts, color='steelblue')
ax.set_title('Samples per Class'); ax.set_xlabel('Class ID'); ax.set_ylabel('Count')
plt.tight_layout(); plt.show()

In [ ]:
# Transforms with augmentation for training, deterministic for eval
MEAN = (0.485, 0.456, 0.406)
STD  = (0.229, 0.224, 0.225)

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomAffine(degrees=12, translate=(0.1, 0.1), scale=(0.9, 1.1), shear=8),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

class WrappedSubset(Dataset):
    def __init__(self, base, indices, transform):
        self.base, self.indices, self.transform = base, list(indices), transform
    def __len__(self):  return len(self.indices)
    def __getitem__(self, i):
        img, lbl = self.base[self.indices[i]]
        return self.transform(img), lbl

class WrappedDataset(Dataset):
    def __init__(self, base, transform):
        self.base, self.transform = base, transform
    def __len__(self):  return len(self.base)
    def __getitem__(self, i):
        img, lbl = self.base[i]
        return self.transform(img), lbl

n = len(trainval_raw)
n_val = int(0.15 * n)
gen = torch.Generator().manual_seed(SEED)
train_idx, val_idx = random_split(range(n), [n - n_val, n_val], generator=gen)

train_ds = WrappedSubset(trainval_raw, train_idx, train_tf)
val_ds   = WrappedSubset(trainval_raw, val_idx,   eval_tf)
test_ds  = WrappedDataset(test_raw, eval_tf)

kw = dict(num_workers=2, pin_memory=True, persistent_workers=True)
train_loader = DataLoader(train_ds, batch_size=64,  shuffle=True,  **kw)
val_loader   = DataLoader(val_ds,   batch_size=256, shuffle=False, **kw)
test_loader  = DataLoader(test_ds,  batch_size=256, shuffle=False, **kw)

print(f'Train: {len(train_ds):,}   Val: {len(val_ds):,}   Test: {len(test_ds):,}')

In [ ]:
# Custom CNN: three Conv-BN-ReLU blocks + GAP + classifier
class TrafficSignCNN(nn.Module):
    def __init__(self, num_classes=43):
        super().__init__()
        def block(in_ch, out_ch):
            return nn.Sequential(
                nn.Conv2d(in_ch,  out_ch, 3, padding=1),
                nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
                nn.Conv2d(out_ch, out_ch, 3, padding=1),
                nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
                nn.MaxPool2d(2), nn.Dropout(0.25),
            )
        self.block1 = block(3, 32)
        self.block2 = block(32, 64)
        self.block3 = block(64, 128)
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 256), nn.BatchNorm1d(256),
            nn.ReLU(inplace=True), nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.block1(x); x = self.block2(x); x = self.block3(x)
        x = self.gap(x)
        return self.classifier(x)

cnn_model = TrafficSignCNN(NUM_CLASSES).to(DEVICE)
print(f'CNN params: {sum(p.numel() for p in cnn_model.parameters()):,}')

In [ ]:
# ResNet-50 with frozen backbone, replace head for 43 classes
resnet_model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
for p in resnet_model.parameters():
    p.requires_grad = False
resnet_model.fc = nn.Linear(resnet_model.fc.in_features, NUM_CLASSES)
resnet_model = resnet_model.to(DEVICE)

total = sum(p.numel() for p in resnet_model.parameters())
trainable = sum(p.numel() for p in resnet_model.parameters() if p.requires_grad)
print(f'ResNet total: {total:,}   trainable: {trainable:,}')

In [ ]:
def train_model(model, model_name, epochs, lr, weight_decay, checkpoint):
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=weight_decay,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    history = {'tr_loss': [], 'tr_acc': [], 'va_loss': [], 'va_acc': []}
    best_val_acc, patience = 0.0, 0
    PATIENCE = 5

    print(f'\nTraining {model_name}  (epochs={epochs}, lr={lr})')
    for ep in range(1, epochs + 1):
        t0 = time.time()
        model.train()
        tr_loss, tr_correct, tr_total = 0.0, 0, 0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            tr_loss    += loss.item() * y.size(0)
            tr_correct += (logits.argmax(1) == y).sum().item()
            tr_total   += y.size(0)

        model.eval()
        va_loss, va_correct, va_total = 0.0, 0, 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                logits = model(x)
                loss = criterion(logits, y)
                va_loss    += loss.item() * y.size(0)
                va_correct += (logits.argmax(1) == y).sum().item()
                va_total   += y.size(0)

        tl, ta = tr_loss / tr_total, tr_correct / tr_total
        vl, va = va_loss / va_total, va_correct / va_total
        history['tr_loss'].append(tl); history['tr_acc'].append(ta)
        history['va_loss'].append(vl); history['va_acc'].append(va)
        scheduler.step()

        flag = ''
        if va > best_val_acc:
            best_val_acc, patience = va, 0
            torch.save(model.state_dict(), checkpoint)
            flag = '  *best'
        else:
            patience += 1

        print(f'Ep {ep:2d} | tr {tl:.4f}/{ta*100:5.2f}% | va {vl:.4f}/{va*100:5.2f}% | {time.time()-t0:4.1f}s{flag}')
        if patience >= PATIENCE:
            print(f'Early stop at epoch {ep}')
            break

    print(f'Best val acc: {best_val_acc*100:.2f}%')
    return history

In [ ]:
history_cnn = train_model(cnn_model, 'Custom CNN',
                          epochs=25, lr=1e-3, weight_decay=1e-4, checkpoint='best_cnn.pt')

history_resnet = train_model(resnet_model, 'ResNet-50',
                             epochs=15, lr=1e-3, weight_decay=1e-4, checkpoint='best_resnet.pt')

In [ ]:
# Training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, key, ylabel, title in [
    (axes[0], 'loss', 'Loss',         'Loss per Epoch'),
    (axes[1], 'acc',  'Accuracy (%)', 'Accuracy per Epoch'),
]:
    cnn_tr, cnn_va = history_cnn[f'tr_{key}'], history_cnn[f'va_{key}']
    res_tr, res_va = history_resnet[f'tr_{key}'], history_resnet[f'va_{key}']
    if key == 'acc':
        cnn_tr = [v*100 for v in cnn_tr]; cnn_va = [v*100 for v in cnn_va]
        res_tr = [v*100 for v in res_tr]; res_va = [v*100 for v in res_va]
    ax.plot(cnn_tr, '--', color='steelblue',  label='CNN train')
    ax.plot(cnn_va,        color='steelblue',  label='CNN val')
    ax.plot(res_tr, '--', color='darkorange', label='ResNet train')
    ax.plot(res_va,        color='darkorange', label='ResNet val')
    ax.set_title(title); ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel)
    ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
def evaluate_on_test(model, checkpoint):
    model.load_state_dict(torch.load(checkpoint, map_location=DEVICE))
    model.eval()
    preds, labels, probs_list = [], [], []
    with torch.no_grad():
        for x, y in test_loader:
            logits = model(x.to(DEVICE))
            probs_list.append(F.softmax(logits, dim=1).cpu().numpy())
            preds.append(logits.argmax(1).cpu().numpy())
            labels.append(y.numpy())
    y_pred = np.concatenate(preds)
    y_true = np.concatenate(labels)
    all_p  = np.concatenate(probs_list)
    top1 = (y_pred == y_true).mean()
    top5 = np.mean([y_true[i] in np.argsort(all_p[i])[-5:] for i in range(len(y_true))])
    return {'y_pred': y_pred, 'y_true': y_true, 'probs': all_p, 'top1': top1, 'top5': top5}

results_cnn    = evaluate_on_test(cnn_model,    'best_cnn.pt')
results_resnet = evaluate_on_test(resnet_model, 'best_resnet.pt')

print(f'{"Metric":<18}{"CNN":>10}{"ResNet":>10}')
print(f'{"Top-1":<18}{results_cnn["top1"]*100:>9.2f}%{results_resnet["top1"]*100:>9.2f}%')
print(f'{"Top-5":<18}{results_cnn["top5"]*100:>9.2f}%{results_resnet["top5"]*100:>9.2f}%')

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, res, name in [
    (axes[0], results_cnn,    f'CNN  (Top-1 {results_cnn["top1"]*100:.2f}%)'),
    (axes[1], results_resnet, f'ResNet  (Top-1 {results_resnet["top1"]*100:.2f}%)'),
]:
    cm = confusion_matrix(res['y_true'], res['y_pred']).astype(float)
    cm /= cm.sum(axis=1, keepdims=True)
    im = ax.imshow(cm, cmap='Blues', vmin=0, vmax=1)
    ax.set_title(name); ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    plt.colorbar(im, ax=ax)
plt.tight_layout(); plt.show()

best_res  = results_resnet if results_resnet['top1'] > results_cnn['top1'] else results_cnn
best_name = 'ResNet-50' if results_resnet['top1'] > results_cnn['top1'] else 'Custom CNN'
print(f'\nPer-class report ({best_name}):')
print(classification_report(best_res['y_true'], best_res['y_pred'],
                            target_names=[n[:28] for n in CLASS_NAMES], digits=3))

In [ ]:
# Sample predictions on the test set
# For each image we print the picture and a caption like:
#   "This sign indicates: Speed limit (70km/h)   (98.4%)"
# Green title = correct prediction, Red title = wrong prediction.

cnn_model.load_state_dict(torch.load('best_cnn.pt',    map_location=DEVICE))
resnet_model.load_state_dict(torch.load('best_resnet.pt', map_location=DEVICE))
cnn_model.eval(); resnet_model.eval()

sample_ids = random.sample(range(len(test_raw)), 8)

def show_predictions(model, model_name, header_color):
    fig, axes = plt.subplots(2, 4, figsize=(16, 9))
    fig.suptitle(f'{model_name} — Sample Test Predictions',
                 fontsize=15, fontweight='bold', color=header_color)
    for ax, idx in zip(axes.flat, sample_ids):
        img_pil, true_label = test_raw[idx]
        x = eval_tf(img_pil).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            probs = F.softmax(model(x), dim=1)[0].cpu().numpy()
        pred = int(probs.argmax())
        conf = float(probs[pred]) * 100
        correct = (pred == true_label)

        ax.imshow(img_pil)
        ax.axis('off')
        caption = f'This sign indicates:\n"{CLASS_NAMES[pred]}"\nConfidence: {conf:.1f}%'
        if not correct:
            caption += f'\n(True: {CLASS_NAMES[true_label]})'
        ax.set_title(caption, fontsize=10,
                     color=('#1e8449' if correct else '#c0392b'),
                     fontweight='bold')
    plt.tight_layout()
    plt.show()

show_predictions(cnn_model,    'Custom CNN', '#2874a6')
show_predictions(resnet_model, 'ResNet-50',  '#b9770e')


In [ ]:
# Final benchmark comparison
benchmarks = {
    'HOG + SVM':            95.71,
    'Human (Stallkamp 2012)': 98.84,
    'Custom CNN':           results_cnn['top1']    * 100,
    'ResNet-50':            results_resnet['top1'] * 100,
}
names  = list(benchmarks.keys())
values = list(benchmarks.values())
colors = ['#95a5a6', '#95a5a6', 'steelblue', 'darkorange']

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(names, values, color=colors, edgecolor='black')
ax.set_xlim(93, 101); ax.set_xlabel('Test Top-1 Accuracy (%)')
ax.set_title('GTSRB Model Comparison')
ax.axvline(98.84, color='grey', linestyle='--', label='Human 98.84%')
for bar, val in zip(bars, values):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
            f'{val:.2f}%', va='center', fontsize=9)
ax.legend()
plt.tight_layout(); plt.show()

print(f'{"":<22}{"CNN":>14}{"ResNet-50":>14}')
print(f'{"Parameters":<22}{sum(p.numel() for p in cnn_model.parameters()):>14,}{sum(p.numel() for p in resnet_model.parameters()):>14,}')
print(f'{"Top-1":<22}{results_cnn["top1"]*100:>13.2f}%{results_resnet["top1"]*100:>13.2f}%')
print(f'{"Top-5":<22}{results_cnn["top5"]*100:>13.2f}%{results_resnet["top5"]*100:>13.2f}%')